## Outline

- Applying our simulation method
    - Differences between means
- Permutation testing - an extended example - mosquitoes and beer (remember these?)
  - the less simplified version
  - if you look at the original paper you'll see that the stats are different than what you get in the example from the textbook (or the video)
  - that is because the textbook/video example uses a simplified measure
  - We will use a real measure - It requires some data wrangling so we see an example of that
  - It also requires more thought connecting the dataset to an outcome measure we can use
  - Our value will be closer to the paper's value
- Python topic: Along the way we will learn about combining dataframes

In [2]:
# Import Numpy library, rename as "np"
import numpy as np
# Import Pandas library, rename as "pd"
import pandas as pd
# Turn on a setting to use Pandas more safely.
pd.set_option('mode.chained_assignment', 'raise')

# Also - set up plotting
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('fivethirtyeight')

## Today's data scientist:  Alan Turing

<img src="AlanTuring.jpg" align="right" width=350>

- Pioneering mathematician and computer scientist
- Produced early theoretical results about computability
  - A Turing machine (what is computable?)
  - The Turing test (can an observer of a conversation between a human and a machine tell which is the machine [forced choice test])
- Formalized concepts of 'algorithm' and 'computation'
- Worked at Bletchley Park on codebreaking
- Designed (with others) a machine to speed up deciphering codes from German Enigma machines
- His efforts (with others on the Bletchley team) were critical to the war effort and especially the battle for the Atlantic.
<br>


- After the war, worked on an early stored-program computer
- Became interested in mathematical biology
- Predicted oscillating chemical reactions that were only observed later (see this incredible story here:  https://philipball.co.uk/wp-content/uploads/2015/05/Turing_long.pdf) and related to markings on animals (like zebras and leopards)
<img src="Leopard.jpg" align="right" width=350>

<br>

- Prosecuted for being a gay man
- Accepted chemical castration rather than prison
- Commited suicide
<br>

- Heartbreaking story
- A tragic example of how limited the gratitude of the state can be
<br>

-Films:  The Imitation Game, Breaking The Code


### Data Science in the news -- 

- Climate change and its effects is a huge area of data science, with many interesting datasets
- Often the questions cross datasets and boundaries
- e.g.
    - Sea level rises
    - What parts of the UK end up flooded with a 1.5 degree change in temperature?
    - What parts are flooded with a 2 degree change in temperature
        - visualization
        - mapping


(https://coastal.climatecentral.org/map/7/3.0868/52.0559/?theme=warming&map_type=decadal_slr&basemap=roadmap&contiguous=true&elevation_model=best_available&esl_model=ipcc_2021&percentile=p50&refresh=true&slr_year=2100&temperature_rise=1.5&temperature_unit=C)
    <br>
    
    - How many people would be displaced by these changes to sea level?
        - Combine sea-level change data with population density maps
(https://luminocity3d.org/MYP1.html#7/51.588/0.111)
<br>
 
- Connections to other current problems
    - Climate change and migration
    - now:  since 2010, 21.5 million people per year are forced to move due to disasters (UN High Commissioner for Refugees - https://news.un.org/en/story/2021/04/1090432)
    - predicted:  31-143 million people per year (up to six times current numbers)
    (https://www.ipcc.ch/report/ar6/wg2/)
<br>


- Conflict

# Permutation test: Another simulation test
# -Mosquitoes and beer-
# Are you more attractive to malaria mosquitoes after a beer? 
### (Remember this? - see video from the beginning of class - week 1, session 2)

Mosquitoes and beer experiment

- Put one person in a tent to capture odor from breath and body
- Open the other tent to the outside air
- Draw air from these two sources into boxes at the end of a Y-junction
- Release mosquitos
- Do they go left to the box connected to the person?
- Do they go right to the box connected to outside air?
<br>   

Experimental design:
- Do this once before drinking anything (control condition)
- Do this again after drinking either beer or water (experimental condition)

# Experimental setup

<img src="AnnotatedExpSetup.png"/>

### A clarification on the design

<img src=beer_water_exp_design.png align="center">

- Beer drinkers are NOT the same as water drinkers
- Before/after drinkers are the same
  - This gives the advantage of removing variability between persons in their intrinsic 'likeability' by mosquitos.  We measure from baseline (before) likeability.
    - beer/water are NOT tested on the same people;  
    - before/after testing IS on the same people
- Beer/water differences **are** measured on **different** people
  - The analogous t-test for beer vs water would be a **two-sample t-test** using difference scores (in case you want to keep track)  
<br>

The full experiment is a mixed design
- before/after is a within participants factor
- water/beer is a between participants factor

### Data

Data are in 'mosquito_beer.csv'
<br>  

Let's read in the data...

In [ ]:
mozzie_data = pd.read_csv("mosquito_beer.csv")
mozzie_data.info() # shows information about columns and column names

In [ ]:
# we should also check that data seem to have been read in ok
mozzie_data.head()

### You need to know about what is in each column
#### consult the source of the data...

- group: beer/water
- test: before/after
- no_odour: air trap
- volunt_odour: person trap
- activated: number of mosquitoes that went towards the traps

### What measure are we interested in?

### Possible  measure

- % of activated mozzies that went toward the volunteer rather than open air
- Take difference between beer/water  


In [ ]:
# Calculate the % of activated mozzies that went to the volunteer

# calculate percent
mozzie_data['percent_act_person'] = mozzie_data['volunt_odour']/mozzie_data['activated']

# Make a data frame with just the values we are interested in.  
# select columns
new_moz_data = mozzie_data[['volunteer','group','test',
                            'no_odour','volunt_odour',
                            'activated','percent_act_person']].copy()
# check the result
new_moz_data.head()

## What about before/after?

- People may differ in how much they attract mosquitos even without drink
- What can you do about that?
<br>

- Remember we would like a single value that tells us about how attractive a person is after they have had either beer or water

### If 'drink' changes preference, then % of activated  mozzies should increase in the 'after' condition
### If 'beer' changes the preference more, then the _increase_ should be bigger for beer compared to water
<br>

What do we need?
- before % for people who will drink water
- after % for people who will drink water
- calculate the difference (after % - before %)
- same for beer people
<br>

- Is the before/after difference bigger for beer?

### Wrangling the data

We don't have before/after on the same row, so we can't do a array subtraction


In [ ]:
# notice that 'before' and 'after' are in separate rows here
new_moz_data

### How do we get before/after data on the same row?

Plan:
- Get water/before data in a data frame
- Get water/after data in a data frame
- combine the two dataframes keeping people matched (this is a database 'merge' or 'join')
<br>

Do the same for beer...

## Put before/after data together...

### (merge two data frames, keeping subjects aligned)

In [ ]:
# can clean up a bit (but don't have to) so we don't end up with so many columns
# in our merged dataframe

# drop some columns we don't need
before_water_data=before_water_data.drop(columns=['test','group']) 
# don't need these columns because the name of the data frame tells us

after_water_data=after_water_data.drop(columns=['test','group']) # don't need this column
# these columns just clutter up the data 
# we add 'before/after' information to the merged data using
# 'suffixes' below

# MERGE

This is a basic data wrangling operation - puts two dataframes together aligning them on one or more columns
- We want to align based on the 'volunteer' column so data from the same person stays together
<br>

### Merge arguments and what they mean:

The merge statement:
```python
water_data = pd.merge(before_water_data,
                      after_water_data,
                      how='outer',
                      on=['volunteer'],
                      suffixes=('_before','_after'))
```

Arguments:
- The first two arguments are the names of the datafames to join: 
    - before_water_data 
    - after_water_data

<img src="join_types.png" align="right" width=300>

- 'how' specifies the type of join
    - 'outer' - keep rows from both dataframes even if rows are missing
        - e.g. if 'after' had a participant missing we would still keep the 'before' entry
    - 'inner' - keep rows that have entries in both dataframes
        - anything missing is removed
    - 'left' - keep the rows from 'before_water_data'
        - anything from 'after' that is not in 'before' is removed
    - 'right' - keep the rows from 'after_water_data'
        - anything from 'before' that is not in 'after' is removed
<br>  


- 'on' says which column to use to align the dataframes
    - e.g. we want to keep the values from the same person aligned
<br>  


- 'suffixes' adds the suffixes to columns from the two dataframes so we know where each column came from
    - there can only be two entries here, one for each dataframe

### A good data science/python cheat sheet

https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

In [ ]:
water_data = pd.merge(before_water_data,after_water_data,
                      how='outer',
                      on=['volunteer'],
                      suffixes=('_before','_after'))
water_data # check there are no misaligined rows (% activated before/after would be missing)

In [ ]:
# How many water drinkers do we have?

len(water_data)

In [ ]:
# Now we can take the difference between before and after

water_data['act_diff']=water_data['percent_act_person_after'] - water_data['percent_act_person_before']
water_data.head()

In [ ]:
# how many beer drinkers do we have?
len(beer_data)

### Let's plot the data and see whether it looks like there is a difference

In [ ]:
bins = np.arange(-1,1,.08) # set up some bins for the bars in our histogram
plt.hist(beer_data["act_diff"],
         bins,
         alpha=0.5,
         color="red",
         label="beer")
plt.hist(water_data["act_diff"],
         bins,
         alpha=0.5,
         color="blue",
         label="water")
plt.xlabel("before/after difference in % preference for volunteers")
plt.ylabel("frequency")
plt.legend()

In [ ]:
mean_beer = beer_data["act_diff"].mean()
mean_water = water_data["act_diff"].mean()
print("mean - beer: ",np.round(mean_beer,4),
      " mean - water:",np.round(mean_water,4))

In [ ]:
real_diff_in_means = mean_beer - mean_water
real_diff_in_means

## Is there a difference - do mozzies like beer drinkers more?

## The traditional approach

- Two means
- Different people in the beer/water conditions
- Do a **two-sample t-test** to decide if the difference in means is reliable
- Assume that there is no difference
- See if that is unlikely given the data

###  Issues: Depends on means from the data being normally distributed
#### (Actually this is a pretty safe assumption)

###  Do you understand what the formula for the t-test is based on?
#### (Remember the 'agonizing pain'?)
###  Understanding depends on knowing about normal distributions and formulas

## The permutation approach

### Use a version of our 'simulation' (or modelling) approach
- Do a simulation lots of times
  - (We will have to decide **What to simulate?**, but never mind that for now)
- Get a value from each simulation
- Store the values

- Compare the distribution of simulated values to the real world value

### What should we simulate?

#### Simulate a world where there is a difference between beer and water?
- How much of a difference?

#### Simulate a world where there is no difference between beer and water?
- If beer/water are not different...
- Labels 'beer' and 'water' are meaningless and just randomly attached to numbers of mosquitos trapped
- We should be able to scramble up the labels and nothing should change
- If we scramble the labels and the real result is different
    - Conclude: Real result is not from a 'no difference' world
    
### We know how to simulate a 'no difference' world.  Let's do that...
### (This is called a 'permutation test' in the literature)

#### 'Permute' means to randomly mix up a set of values (but not add, subtract or repeat any)
#### (technically, a 'permutation' is one of the many possible orderings of the numbers in the data)

## Permutation in pictures

##  The real world - beer is gold, water is blue, number is the number of mosquitoes trapped 

### (NOTE: our numbers are differences in % so they look a bit different than these)

<img src="RealWorldPic.png"/>

### The beer mean

<img src="BeerMeanPic.png"/>

### The water mean

<img src="WaterMeanPic.png"/>

### These are the real world values

## The simulated world

### Shuffle the values

<img src="ShufflePic.png"/>

### The simulated beer mean

<img src="SimulatedBeerPic.png"/>

#### Uses the first 25 values (remember above how we listed the number of 'beer' values?)

### The simulated water mean

<img src="SimulatedWaterPic.png"/>

#### Uses the last 18 values

### Remember:  If beer/water makes no difference the colour of the balls is meaningless
### Another way to think about this is that the proportion of blue and gold balls is about the same in the "beer" and "water" sample
- If there is was a difference in the real sample, there are still about the same number of higher and lower values in each of the scrambled samples

### These are one set of simulated values (we will want to do this lots of times)

### The plan:
### Do the simulation lots of times

### For each simulation
- Shuffle values
- Select the shuffled values with the 'beer' label
- Select the shuffled values with the 'water' label
- Calculate the difference between beer and water shuffled value means
- Store the difference
- Repeat many times

### Compared the set of stored differences to the one real world difference
### Is the real world difference rare in our 'no difference' world?

### PYTHON CODE
### pd.concat

- put together dataframes that have the same column structure
- essentially pasting one at the bottom of the other

In [ ]:
# put both beer and water in a single dataframe (so we can shuffle the act_diff values)

# label the values from each dataframe before we combine them
# (so we know where the values came from when we put them together)
beer_data["condition"]="beer"
water_data["condition"]="water"

mozzie_diffs = pd.concat([beer_data,water_data])
# NOTICE - dataframes are inside []
# you might want to combine more than just two, 
# so we make a list of all of the dataframes we want to combine

mozzie_diffs.head()

In [ ]:
# Plan our  loop



### How do we shuffle values?

**Remember:** `np.random.permutation(some_np.array)`


In [ ]:
# Do one simulation 

# shuffle the values
diff_values = np.random.permutation(np.array(mozzie_diffs["act_diff"]))
# make a boolean array to help pick out the 'beer' values
is_beer = mozzie_diffs["condition"] == "beer"
# make a boolean array to help pick out the 'water' values
is_water = mozzie_diffs["condition"] == "water"

# select the beer values
beer_diffs = diff_values[is_beer]
# select the water values
water_diffs = diff_values[is_water]

# calculate means and take the difference
sim_diff=beer_diffs.mean()-water_diffs.mean()

# this value should be just above or below zero
sim_diff

#### You might want to turn this into a function

In [ ]:
# Put your simulation code together here

In [ ]:
# calculate p-value

# How many of the 'no difference world' values were 
# as big or bigger than the real world value?
p_value = np.count_nonzero(sim_diff >= real_diff_in_means) / num_simulations
p_value

### What is  your conclusion?

p_value is the probability that we are in the 'no_difference' world.

Is the real world value likely or rare in the 'no_difference' world?

# More practice (if you want to practice, try this...[extra exercise!])


## NHS waiting times - have they increased?

## Question:  Were NHS waiting times longer in 2020 than in 2012?

### Comes from real world dataset

     https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/
     
     
### Load the file

```python
wait_times = pd.read_csv("NHSWaitingTimes.csv")
```

### Important columns

- Year (2020 vs 2012) column name: "year"
- Month (December 2020, December 2012, September 2020) column name: "month"
- Median waiting time.  Column name: "median_waits"
- Longest waiting times.  Column name: "longest_waits"
- Don't worry about "long_wait_percent" -- this is the percentage used to cut off the longest times and it has changed a bit, but we don't have to worry about that difference here

### Answer the question:  Have median waiting times increased? (note we are only interested in one direction)

### If you want more practice answer whether the longest times have increased
### Note that these answers don't have to be the same

In [3]:
wait_times = pd.read_csv("NHSWaitingTimes.csv")
wait_times.head()

,year,month,type_of_treatment,median_waits,longest_waits,long_wait_percent
0,2020,December,General Surgery,11.9,49.1,92
1,2020,December,Urology,11.7,48.2,92
2,2020,December,Trauma & Orthopaedics,14.6,52.0,92
3,2020,December,"Ear, Nose & Throat (ENT)",14.9,51.1,92
4,2020,December,Ophthalmology,11.9,47.9,92
